##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: All about tokens

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Counting_Tokens.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

An understanding of tokens is central to using the Gemini API. This guide will provide a interactive introduction to what tokens are and how they are used in the Gemini API.

## About tokens

LLMs break up their input and produce their output at a granularity that is smaller than a word, but larger than a single character or code-point.

These **tokens** can be single characters, like `z`, or whole words, like `the`. Long words may be broken up into several tokens. The set of all tokens used by the model is called the vocabulary, and the process of breaking down text into tokens is called tokenization.

For Gemini models, a token is equivalent to about 4 characters. **100 tokens are about 60-80 English words**.

When billing is enabled, the price of a paid request is controlled by the [number of input and output tokens](https://ai.google.dev/pricing), so knowing how to count your tokens is important.


## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai).

In [ ]:
%pip install -q -U "google-genai>=1.0.0"

### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [ ]:
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')

### Initialize SDK client

With the new SDK you now only need to initialize a client with you API key (or OAuth if using [Vertex AI](https://cloud.google.com/vertex-ai)). The model is now set in each call.

In [ ]:
from google import genai

client = genai.Client(api_key=GOOGLE_API_KEY)

## Tokens in the Gemini API

### Context windows

The models available through the Gemini API have context windows that are measured in tokens. These define how much input you can provide, and how much output the model can generate, and combined are referred to as the "context window". This information is available directly through [the API](https://ai.google.dev/api/rest/v1/models/get) and in the [models](https://ai.google.dev/models/gemini) documentation.

In this example you can see the `gemini-2.5-flash` model has an 1M tokens context window. If you need more, Pro models have an even bigger 2M tokens context window.

In [ ]:
model_info = client.models.get(model=MODEL_ID)

print("Context window:",model_info.input_token_limit, "tokens")
print("Max output window:",model_info.output_token_limit, "tokens")

## Counting tokens

The API provides an endpoint for counting the number of tokens in a request: [`client.models.count_tokens`](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens). You pass the same arguments as you would to [`client.models.generate_content`](https://googleapis.github.io/python-genai/#generate-content) and the service will return the number of tokens in that request.

### Choose a model

Now select the model you want to use in this guide, either by selecting one in the list or writing it down. Keep in mind that some models, like the 2.5 ones are thinking models and thus take slightly more time to respond (cf. [thinking notebook](./Get_started_thinking.ipynb) for more details and in particular learn how to switch the thiking off).

The tokenization should be more or less the same for each of the Gemini models, but you can still switch between the different ones to double-check.

For more information about all Gemini models, check the [documentation](https://ai.google.dev/gemini-api/docs/models/gemini) for extended information on each of them.

In [ ]:
MODEL_ID = "gemini-3-flash-preview" # @param ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.1-flash-lite-preview", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {"allow-input":true, isTemplate: true}

### Text tokens

In [ ]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents="What's the highest mountain in Africa?",
)
print("Prompt tokens:",response.total_tokens)

When you call `client.models.generate_content` (or `chat.send_message`) the response object has a `usage_metadata` attribute containing both the input, output, and thinking token counts (`prompt_token_count`, `candidates_token_count` and `thoughts_token_count`):

In [ ]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="The quick brown fox jumps over the lazy dog."
)
print(response.text)

In [ ]:
print("Prompt tokens:\t ",response.usage_metadata.prompt_token_count)
print("Thinking tokens:",response.usage_metadata.thoughts_token_count)
print("Output tokens:\t ",response.usage_metadata.candidates_token_count)
print("--------------")
print("Total tokens:\t",response.usage_metadata.total_token_count)

In case you are using [caching](./Caching.ipynb#scrollTo=t_PWabuayrf-), the number of cached token will be indicated in `response.usage_metadata.cached_content_token_count`.

### Multi-modal tokens

All input to the API is tokenized, including images or other non-text modalities.

Images are considered to be a fixed size, so they consume a fixed number of tokens, regardless of their display or file size.

Video and audio files are converted to tokens at a fixed per second rate.

The current rates and token sizes can be found on the [documentation](https://ai.google.dev/gemini-api/docs/tokens?lang=python#multimodal-tokens)

In [ ]:
!curl -L https://goo.gle/instrument-img -o organ.jpg

In [ ]:
import PIL
from IPython.display import display, Image

organ = PIL.Image.open('organ.jpg')
display(Image('organ.jpg', width=300))

#### Inline content

Media objects can be sent to the API inline with the request:

In [ ]:
response = client.models.count_tokens(
    model=MODEL_ID,
    contents=[organ]
)

print("Prompt with image tokens:",response.total_tokens)

You can try with different images and should always get the same number of tokens, that is independent of their display or file size. Note that an extra token seems to be added, representing the empty prompt.

#### Files API

The model sees identical tokens if you upload parts of the prompt through the files API instead:

In [ ]:
organ_upload = client.files.upload(file='organ.jpg')

response = client.models.count_tokens(
    model=MODEL_ID,
    contents=organ_upload,
)

print("Prompt with image tokens:",response.total_tokens)

Audio and video are each converted to tokens at a fixed rate of tokens per minute.

In [ ]:
!curl -q -o sample.mp3  "https://storage.googleapis.com/generativeai-downloads/data/State_of_the_Union_Address_30_January_1961.mp3"
!ffprobe -v error -show_entries format=duration sample.mp3

As you can see, this audio file is 2610s long.

In [ ]:
audio_sample = client.files.upload(file='sample.mp3')

response = client.models.count_tokens(
    model=MODEL_ID,
    contents=audio_sample
)

print("Prompt with audio tokens:",response.total_tokens)
print("Tokens per second: ",response.total_tokens/2610)

As you can see this corresponds to about 32 tokens per second of audio.

### Chat, tools and caching

Chat, tools and caching are currently not supported by the unified SDK `count_tokens` method. This notebook will be updated when that will be the case.

In the meantime you can still check the token used after the call using the `usage_metadata` from the response. Check the [caching notebook](./Caching.ipynb#scrollTo=t_PWabuayrf-) for an example.

## Further reading

For more on token counting, check out the [documentation](https://ai.google.dev/gemini-api/docs/tokens?lang=python#multimodal-tokens) or the API reference:

* [`countTokens`](https://ai.google.dev/api/rest/v1/models/countTokens) REST API reference,
* [`count_tokens`](https://googleapis.github.io/python-genai/#count-tokens-and-compute-tokens) Python API reference,

In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Vertex AI Count Tokens

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/jbrache/vertex-ai-things/blob/main/gen-ai/quickstarts/Counting_Tokens.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2Fjbrache%2Fvertex-ai-things%2Fmain%2Fgen-ai%2Fquickstarts%2FCounting_Tokens.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/jbrache/vertex-ai-things/main/gen-ai/quickstarts/Counting_Tokens.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/jbrache/vertex-ai-things/blob/main/gen-ai/quickstarts/Counting_Tokens.ipynb">
      <img width="32px" src="https://storage.googleapis.com/github-repo/generative-ai/logos/GitHub_Invertocat_Dark.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

The [CountTokens API](https://docs.cloud.google.com/vertex-ai/generative-ai/docs/model-reference/count-tokens) calculates the number of input tokens before sending a request to the Gemini API.

Use the CountTokens API to prevent requests from exceeding the model context window, and estimate potential costs based on billable characters or tokens.

The CountTokens API can use the same `contents` parameter as Gemini API inference requests.

### Install Google Gen AI SDK and other required packages


In [ ]:
%pip install --upgrade --quiet google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.6/760.6 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.7/240.7 kB 15.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.49.1 which is incompatible.


### Authenticate your notebook environment

If you are running this notebook in **Google Colab**, run the cell below to authenticate your account.

In [ ]:
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Set Google Cloud project information

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
import os

from google import genai

# fmt: off
PROJECT_ID = "the-foo-bar"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
LOCATION = "global" # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on

if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))
if not LOCATION:
    LOCATION = os.environ.get("GOOGLE_CLOUD_REGION")

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

### Import libraries

In [ ]:
from IPython.display import Markdown, display
from google import genai
from google.genai.types import GenerateContentConfig, HttpOptions, Part, MediaModality

# [TODO]: Add other library imports here

### Load model

In [ ]:
MODEL_ID = "gemini-3.1-pro-preview"  # @param {type:"string"}

## Audio Understanding

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, http_options=HttpOptions(api_version="v1"))
prompt = """
Transcribe the interview, in the format of timecode, speaker, caption.
Use speaker A, speaker B, etc. to identify speakers.
"""
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        prompt,
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/audio/pixel.mp3",
            mime_type="audio/mpeg",
        ),
    ],
    # Required to enable timestamp understanding for audio-only files
    config=GenerateContentConfig(audio_timestamp=True),
)
print(response.text)

00:00 Speaker B: Your devices are getting better over time, and so we think about it across the entire portfolio, from phones, to watch, to buds, to tablet. We get really excited about how we can tell a joint narrative across everything.

00:14 Speaker A: Welcome to the Made by Google podcast, where we meet the people who work on the Google products you love. Here's your host, Rashid Finch.

00:22 Speaker A: Today we're talking to Aisha Sharif and DeCarlos Love. They're both product managers for various Pixel devices and work on something that all the Pixel owners love, the Pixel feature drops.

00:35 Speaker A: This is the Made by Google podcast. 

00:37 Speaker A: Aisha, which feature on your Pixel phone has been most transformative in your own life?

00:43 Speaker B: So many features. I am a singer, so I actually think Recorder transcription has been incredible. Because before I would record songs, I'd just like freestyle them, record them, type them up. But now with transcription, 

In [ ]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  cache_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=31
    ),
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=15463
    ),
  ],
  cached_content_token_count=15494,
  candidates_token_count=2915,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=2915
    ),
  ],
  prompt_token_count=15752,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=15720
    ),
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=32
    ),
  ],
  thoughts_token_count=7943,
  total_token_count=26610,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
)

### Count Tokens

In [ ]:
!curl -q -o sample.mp3  "https://storage.googleapis.com/cloud-samples-data/generative-ai/audio/pixel.mp3"
!ffprobe -v error -show_entries format=duration sample.mp3

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4912k  100 4912k    0     0  31.7M      0 --:--:-- --:--:-- --:--:-- 31.7M
[FORMAT]
duration=628.793375
[/FORMAT]


In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, http_options=HttpOptions(api_version="v1"))

In [ ]:
contents = [
    Part.from_uri(
        file_uri="gs://cloud-samples-data/generative-ai/audio/pixel.mp3",
        mime_type="audio/mpeg",
    )
]

response = client.models.count_tokens(
    model=MODEL_ID,
    contents=contents,
)

In [ ]:
response.total_tokens

15700

In [ ]:
print("Prompt with audio tokens:",response.total_tokens)
print("Tokens per second: ",response.total_tokens/628)

Prompt with audio tokens: 15700
Tokens per second:  25.0


## Video Understanding (Video Only)

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, http_options=HttpOptions(api_version="v1"))
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/video/ad_copy_from_video.mp4",
            mime_type="video/mp4",
        ),
        "What is in the video?",
    ],
)
print(response.text)

A person is surfing.


In [ ]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=5,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=5
    ),
  ],
  prompt_token_count=1158,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.VIDEO: 'VIDEO'>,
      token_count=1152
    ),
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=6
    ),
  ],
  thoughts_token_count=229,
  total_token_count=1392,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
)

### Count Tokens

In [ ]:
!curl -q -o sample.mp4  "https://storage.googleapis.com/cloud-samples-data/generative-ai/video/ad_copy_from_video.mp4"
!ffprobe -v error -show_entries format=duration sample.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3676k  100 3676k    0     0  26.1M      0 --:--:-- --:--:-- --:--:-- 26.4M
[FORMAT]
duration=18.101417
[/FORMAT]


In [ ]:
contents = [
    Part.from_uri(
        file_uri="gs://cloud-samples-data/generative-ai/video/ad_copy_from_video.mp4",
        mime_type="video/mp4",
    )
]

response = client.models.count_tokens(
    model=MODEL_ID,
    contents=contents,
)

In [ ]:
print("Prompt with video tokens:",response.total_tokens)
print("Tokens per second: ",response.total_tokens/18)

Prompt with video tokens: 1350
Tokens per second:  75.0


## Video Understanding (Video + Audio)

In [ ]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION, http_options=HttpOptions(api_version="v1"))
response = client.models.generate_content(
    model=MODEL_ID,
    contents=[
        Part.from_uri(
            file_uri="gs://cloud-samples-data/generative-ai/video/pixel8.mp4",
            mime_type="video/mp4",
        ),
        "What is in the video?",
    ],
)
print(response.text)

A young woman is seen walking in the video. She is recording a video of the view. There are different views of the street. She is making a video from Google Pixel 8 Pro. It is night time. She is recording using the video boost feature in low light.


In [ ]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=56,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=56
    ),
  ],
  prompt_token_count=5085,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.VIDEO: 'VIDEO'>,
      token_count=3648
    ),
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=6
    ),
    ModalityTokenCount(
      modality=<MediaModality.AUDIO: 'AUDIO'>,
      token_count=1431
    ),
  ],
  thoughts_token_count=775,
  total_token_count=5916,
  traffic_type=<TrafficType.ON_DEMAND: 'ON_DEMAND'>
)

In [ ]:
!curl -q -o sample.mp4  "https://storage.googleapis.com/cloud-samples-data/generative-ai/video/pixel8.mp4"
!ffprobe -v error -show_entries format=duration sample.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4835k  100 4835k    0     0  32.5M      0 --:--:-- --:--:-- --:--:-- 32.7M
[FORMAT]
duration=57.208000
[/FORMAT]


In [ ]:
total_tokens = 0
duration = 57
for detail in response.usage_metadata.prompt_tokens_details:
  if detail.modality == 'AUDIO':
    print("Audio Tokens per second: ",detail.token_count/duration)
    total_tokens += detail.token_count
  if detail.modality == 'VIDEO':
    print("Video Tokens per second: ",detail.token_count/duration)
    total_tokens += detail.token_count

print("Total Video + Audio tokens: ", total_tokens)

Video Tokens per second:  64.0
Audio Tokens per second:  25.105263157894736
Total Video + Audio tokens:  5079


### Count Tokens

In [ ]:
!curl -q -o sample.mp4  "https://storage.googleapis.com/cloud-samples-data/generative-ai/video/pixel8.mp4"
!ffprobe -v error -show_entries format=duration sample.mp4

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 4835k  100 4835k    0     0  11.7M      0 --:--:-- --:--:-- --:--:-- 11.6M
[FORMAT]
duration=57.208000
[/FORMAT]


In [ ]:
contents = [
    Part.from_uri(
        file_uri="https://storage.googleapis.com/cloud-samples-data/generative-ai/video/pixel8.mp4",
        mime_type="video/mp4",
    )
]

count_tokens_response = client.models.count_tokens(
    model=MODEL_ID,
    contents=contents,
)

In [ ]:
count_tokens_response

CountTokensResponse(
  sdk_http_response=HttpResponse(
    headers=<dict len=9>
  ),
  total_tokens=5985
)

In [ ]:
print("Prompt with video tokens:",count_tokens_response.total_tokens)
print("Tokens per second: ",count_tokens_response.total_tokens/57)

Prompt with video tokens: 5985
Tokens per second:  105.0
